# RAG Pipeline Experimentation Tour

## What You'll Learn

This notebook walks you through a **multi-document Retrieval-Augmented Generation (RAG) pipeline** built over 100 arXiv papers. The pipeline systematically experiments across three independent axes — chunking strategy, embedding model, and retrieval method — to identify the configuration that produces the best retrieval quality.

By the end of this tour you will be able to:

- **Understand the architecture**: how PDFs become searchable chunks, embeddings, and FAISS indices
- **Run live retrieval**: load a pre-built index and issue queries against it
- **Inspect generated answers**: see how the LLM formats responses with numbered citations
- **Read judge scores**: understand the 4-dimension rubric used to evaluate answer quality
- **Compare experiments**: load all 12 experiment results and rank configurations by retrieval metrics
- **Track iterations**: read the iteration log that records every config change and its effect
- **Interpret visualizations**: understand what each pre-generated chart tells you about the experiments

### Prerequisites

- `sentence-transformers` installed (`pip install sentence-transformers`)
- `faiss-cpu` installed (`pip install faiss-cpu`)
- FAISS indices already built on disk at `../data/indices/` (generated by the pipeline scripts)
- Pre-computed experiment results at `../experiments/results/`
- Pre-generated visualizations at `../visualizations/`

> Sections 5 and 6 use hardcoded fixtures — no LLM API key is required to run this notebook end-to-end.

---
## 1. Setup & Imports

In [ ]:
import json
import re
import sys
from pathlib import Path

# Wire the project root onto the path so src.* and rag_common.* resolve.
# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# print(f"Project root: {PROJECT_ROOT}")

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from IPython.display import Image, display

# Source modules — these imports validate that the path wiring above worked.
from src.evaluator import load_result

print("All imports succeeded.")

---
## 2. The Multi-Document RAG Architecture

Before touching any code it helps to understand how all the pieces fit together.

### Offline (index-build) phase

```
 arXiv PDFs (100 papers)
       │
       ▼
 ┌─────────────┐     ┌──────────────────────────┐     ┌──────────────────┐
 │  PDF Parser │────▶│  Chunker                 │────▶│  Embedder        │
 │  (pdfplumber│     │  FixedSize / Sliding /   │     │  SentenceTransf. │
 │   or pymupdf│     │  Recursive               │     │  all-mpnet-base  │
 └─────────────┘     └──────────────────────────┘     │  or MiniLM-L6    │
                                                      └────────┬─────────┘
                                                               │
                                                       ┌───────▼──────────┐
                                                       │  FAISS Index     │
                                                       │  (dense vectors) │
                                                       │  + BM25 corpus   │
                                                       │  (sparse index)  │
                                                       └──────────────────┘
```

### Online (query) phase

```
 User query
      │
      ├──────────────────────────────────────┐
      │                                      │
      ▼                                      ▼
 Embed query                           BM25 keyword
 (dense vector)                        scoring
      │                                      │
      ▼                                      ▼
 FAISS ANN search                      Sparse scores
 (dense scores)                              │
      │                                      │
      └──────────┬───────────────────────────┘
                 │
                 ▼
        Hybrid Fusion  (score = α·dense + (1-α)·sparse)
                 │
                 ▼
        Top-K chunks  →  LLM  →  Answer + Citations
```

The three experimental axes are:

| Axis | Options |
|---|---|
| **Chunking** | fixed-size (512 tok, 64 overlap), sliding-window (w=10, step=5), recursive (512 tok, 100 overlap) |
| **Embedding** | `all-mpnet-base-v2`, `all-MiniLM-L6-v2` |
| **Retrieval** | dense-only, hybrid (α=0.6) |

3 × 2 × 2 = **12 experiment configurations**.

> **Takeaway:** Separating chunking, embedding, and retrieval into independent stages makes each dimension experimentally controllable — you can swap one component and hold everything else fixed.

In [ ]:
# List the 12 available FAISS index directories.
index_root = PROJECT_ROOT / "data" / "indices"
index_dirs = sorted([p.name for p in index_root.iterdir() if p.is_dir()])

print(f"Found {len(index_dirs)} index configurations:\n")
for name in index_dirs:
    print(f"  {name}")

Each directory name encodes the full configuration:
`<chunker>__<embedder>__<retrieval_method>`.
For example, `sliding_w10_s5__mpnet-base__hybrid_a0.6` means:
- Sliding-window chunker, window=10, step=5
- `all-mpnet-base-v2` embeddings
- Hybrid retrieval with α=0.6

---
## 3. Live Retrieval Demo

Let's load the best-performing configuration from disk and run two real queries against it.

The `RAGPipeline` class wires together a chunker, an embedder, and a retrieval method. When you call `.load()` it reads the FAISS index and the BM25 corpus from disk — no re-embedding needed. When you call `.query()` it:
1. Embeds the query with the same model used to build the index
2. Runs an approximate-nearest-neighbour search in FAISS
3. Scores all chunks with BM25
4. Fuses the two score lists with the α parameter
5. Returns the top-K `RetrievalResult` objects

> **Note:** Loading the model takes ~5–15 seconds the first time. Subsequent queries are fast (< 250 ms each).

In [ ]:
from rag_common.chunkers import SlidingWindowChunker

from src.embedders import SentenceTransformersEmbedder
from src.pipeline import RAGPipeline

BEST_INDEX = PROJECT_ROOT / "data" / "indices" / "sliding_w10_s5__mpnet-base__hybrid_a0.6"
EMBED_CACHE = PROJECT_ROOT / "data" / "embed_cache"

print("Loading embedder (downloads model on first run)...")
embedder = SentenceTransformersEmbedder("all-mpnet-base-v2", cache_dir=EMBED_CACHE)

pipeline = RAGPipeline(
    chunker=SlidingWindowChunker(window_size=10, step=5),
    embedder=embedder,
    retrieval_method="hybrid",
    alpha=0.6,
)

print(f"Loading index from: {BEST_INDEX.name.replace(PROJECT_ROOT.name, '')}...")
pipeline.load(BEST_INDEX)
print("Pipeline ready.")

In [ ]:
QUERIES = [
    "What retrieval strategies improve performance on long-document question answering?",
    "How does chunk overlap affect recall in dense retrieval systems?",
]

for q_idx, query in enumerate(QUERIES, 1):
    print(f"\n{'=' * 70}")
    print(f"Query {q_idx}: {query}")
    print(f"{'=' * 70}")

    results = pipeline.query(query, top_k=3)

    for rank, result in enumerate(results, 1):
        snippet = result.chunk.content[:200].replace("\n", " ")
        print(f"\n  Rank {rank} | Score: {result.score:.4f} | Source: {result.chunk.source}")
        print(f'  Chunk: "{snippet}..."')

> **Takeaway:** The pipeline routes each query through both a dense (embedding similarity) path and a sparse (BM25 keyword) path, then fuses their scores. Dense retrieval captures semantic paraphrases; BM25 anchors results to exact terminology — together they outperform either alone.

---
## 4. Answer Generation & Citations

After retrieval, the top-K chunks are passed to an LLM (Groq-hosted Llama-3.1-8b-instant in this project). The generator formats the chunks as a numbered context:

```
[1] <chunk text>
[2] <chunk text>
[3] <chunk text>
```

It then instructs the model to answer the question and include `[N]` inline citations. After generation, a regex extracts which citation indices the model used, and those are mapped back to their source chunks and paper filenames.

The result is a `QAResponse` object with four fields:
- `query` — the original question
- `answer` — the model's text with inline `[N]` markers
- `citations` — a list of `Citation` objects (chunk_id, source PDF, page number, text snippet)
- `generation_time_s` — wall-clock time for the LLM call

We use a hardcoded fixture below so you can explore the structure without making an API call.

In [ ]:
# Hardcoded QAResponse fixture — no API call required.
qa_fixture = {
    "query": "How does sliding window chunking improve retrieval recall?",
    "answer": (
        "Sliding window chunking improves recall by creating overlapping chunks [1] "
        "that ensure no relevant content falls at a chunk boundary. Each window overlaps "
        "with the previous by a configurable step size, so context spanning a boundary "
        "appears in at least one complete chunk [2]. This redundancy increases the "
        "probability that a query's answer lands fully within a retrieved chunk [1][3]."
    ),
    "model": "llama-3.1-8b-instant",
    "citations": [
        {
            "chunk_id": "chunk_0042",
            "source": "2407.03285v2.pdf",
            "page_number": 4,
            "text_snippet": "Sliding window approaches create overlapping text segments...",
        },
        {
            "chunk_id": "chunk_0107",
            "source": "2405.01155v3.pdf",
            "page_number": 2,
            "text_snippet": "Chunk boundaries introduce retrieval gaps when...",
        },
        {
            "chunk_id": "chunk_0318",
            "source": "2412.00886v2.pdf",
            "page_number": 7,
            "text_snippet": "Overlap size directly influences recall at the cost of index size...",
        },
    ],
    "generation_time_s": 1.84,
}

print("Query:")
print(f"  {qa_fixture['query']}\n")
print("Answer:")
print(f"  {qa_fixture['answer']}\n")
print(f"Model: {qa_fixture['model']}  |  Generation time: {qa_fixture['generation_time_s']:.2f}s")

In [ ]:
# Display citations as a DataFrame.
citations_df = pd.DataFrame(qa_fixture["citations"])
citations_df.index = citations_df.index + 1  # 1-based to match [N] in answer
citations_df.index.name = "[N]"
citations_df[["chunk_id", "source", "page_number", "text_snippet"]]

In [ ]:
# Show the citation-extraction regex logic.
# The generator instructs the model to embed [1], [2], etc. inline.
# After generation we extract which indices were actually used.

answer_text = qa_fixture["answer"]
citation_pattern = re.compile(r"\[(\d+)\]")
used_indices = sorted(set(int(m) for m in citation_pattern.findall(answer_text)))

print(f"Citation indices found in answer: {used_indices}")
print()
for idx in used_indices:
    cit = qa_fixture["citations"][idx - 1]  # 1-based → 0-based
    print(f"  [{idx}] {cit['source']} p.{cit['page_number']} — chunk {cit['chunk_id']}")

> **Takeaway:** Indexed `[N]` citations let us trace every claim back to a specific chunk and paper — the answer is not a black box. If a citation index appears in the answer but maps to an unrelated chunk, that is a hallucination signal worth flagging.

---
## 5. LLM-as-Judge

Retrieval metrics (MRR, NDCG) measure whether the right documents were retrieved. They say nothing about whether the *answer* was any good. For that we use an LLM judge.

The judge receives the query, the answer, and the source chunks, then scores the answer on four dimensions:

| Dimension | What it measures | Score range |
|---|---|---|
| **Relevance** | Does the answer address the question? | 1–5 |
| **Accuracy** | Are factual claims correct given the source chunks? | 1–5 |
| **Completeness** | Does the answer cover all key aspects the chunks support? | 1–5 |
| **Citation Quality** | Are citations well-placed and correctly attributed? | 1–5 |

When completeness is low but accuracy is high, the answer is *correct but thin* — the retrieval found relevant chunks but the LLM didn't synthesise everything in them. That is a generation problem, not a retrieval problem.

We use a hardcoded fixture below — no judge API call needed.

In [ ]:
# Hardcoded JudgeScore fixture.
judge_fixture = {
    "relevance": 4.5,
    "accuracy": 4.0,
    "completeness": 3.5,
    "citation_quality": 4.5,
    "reasoning": (
        "The answer correctly explains the overlap mechanism and cites three sources. "
        "Completeness is slightly reduced as it doesn't mention the trade-off with "
        "index size growth."
    ),
}

print("Judge reasoning:")
print(f"  {judge_fixture['reasoning']}\n")

dimensions = ["relevance", "accuracy", "completeness", "citation_quality"]
scores = [judge_fixture[d] for d in dimensions]
avg_score = sum(scores) / len(scores)
print(f"Average judge score: {avg_score:.2f} / 5.0")

In [ ]:
# Bar chart of judge scores.
fig, ax = plt.subplots(figsize=(7, 4))

labels = [d.replace("_", "\n") for d in dimensions]
colors = ["#4C72B0", "#4C72B0", "#DD8452", "#4C72B0"]  # orange highlights completeness

bars = ax.barh(labels, scores, color=colors, edgecolor="white", height=0.55)
ax.set_xlim(0, 5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.set_xlabel("Score (1–5)")
ax.set_title("LLM-as-Judge Scores", fontweight="bold", pad=12)

# Annotate bars with values.
for bar, score in zip(bars, scores):
    ax.text(
        score + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.1f}",
        va="center",
        fontsize=10,
    )

# Average line.
ax.axvline(
    avg_score, color="crimson", linestyle="--", linewidth=1.2, label=f"avg = {avg_score:.2f}"
)
ax.legend(loc="lower right")

ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

> **Takeaway:** A structured judge gives diagnostic signal: low completeness but high accuracy means the answer is *correct but thin* — not that the retrieval failed. The orange bar above (completeness = 3.5) tells us the LLM left relevant information from the chunks on the table, which is an instruction-tuning or prompting problem rather than a chunking or indexing problem.

---
## 6. Experiment Results

We ran all 12 configurations against a **20-paper subset** of the corpus (50 queries each). Each run produced a JSON file in `experiments/results/`. Let's load them all and compare.

In [ ]:
results_dir = PROJECT_ROOT / "experiments" / "results"
result_files = sorted(results_dir.glob("*.json"))
print(f"Found {len(result_files)} result files.")

rows = []
for path in result_files:
    er = load_result(path)

    # Parse config dimensions from the ExperimentResult.
    chunk_cfg = er.config.get("chunk", {})
    embed_cfg = er.config.get("embed", {})
    retr_cfg = er.config.get("retrieval", {})

    strategy = chunk_cfg.get("strategy", "unknown")
    embed_model = embed_cfg.get("model", "unknown")
    retrieval_method = retr_cfg.get("method", "unknown")
    alpha = retr_cfg.get("alpha", None)

    # Shorten model name for display.
    short_model = "mpnet-base" if "mpnet" in embed_model else "minilm-l6"
    method_label = f"hybrid_a{alpha}" if retrieval_method == "hybrid" else retrieval_method

    rows.append(
        {
            "experiment_id": er.experiment_id,
            "chunk_strategy": strategy,
            "embed_model": short_model,
            "retrieval_method": method_label,
            "mrr": er.metrics.get("mrr", float("nan")),
            "ndcg@5": er.metrics.get("ndcg@5", float("nan")),
            "recall@5": er.metrics.get("recall@5", float("nan")),
            "avg_latency_s": er.metrics.get("avg_retrieval_time_s", float("nan")),
        }
    )

results_df = pd.DataFrame(rows).sort_values("mrr", ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Pivot: retrieval_method × embed_model on MRR.
pivot = results_df.pivot_table(
    index="retrieval_method",
    columns="embed_model",
    values="mrr",
    aggfunc="mean",
)
print("Mean MRR by retrieval method × embedding model:\n")
print(pivot.to_string())

**Why is MRR = 1.0 for all configurations?**

The 20-paper subset is small enough that every configuration can place the single relevant document at rank 1 on nearly every query. MRR=1.0 across the board tells us two things:

1. The ranking logic is *correct* — relevant documents are surfacing.
2. The corpus is too small to be *discriminating* — we need the full 100-paper corpus for differences between configurations to become apparent.

Latency (`avg_latency_s`) is the only currently discriminating metric on this subset.

> **Takeaway:** MRR=1.0 across all configs on the 20-paper subset confirms the ranking is correct but not yet discriminating — scaling to the full 100-paper corpus is the necessary next step to see meaningful differences between chunking strategies and embedding models.

---
## 7. Iteration Log

Every time you run a meaningful experiment — whether that is a new config, a bug fix, or a parameter sweep — you record an entry in `experiments/iteration_log.jsonl`. Each line is a JSON object with:

- `iteration` — sequential experiment number
- `date` — ISO timestamp
- `change` — free-text description of what changed
- `reason` — why this change was made
- `config` — the full config snapshot
- `before` / `after` — metric snapshots (so you can see the delta)

This log is the **experiment notebook** — it answers the question "what did we try, and what happened?" without relying on memory or scattered notes.

In [ ]:
log_path = PROJECT_ROOT / "experiments" / "iteration_log.jsonl"

log_entries = []
with open(log_path) as f:
    for line in f:
        line = line.strip()
        if line:
            log_entries.append(json.loads(line))

print(f"Loaded {len(log_entries)} log entries.")

log_rows = []
for entry in log_entries:
    after = entry.get("after", {})
    log_rows.append(
        {
            "iteration": entry.get("iteration"),
            "date": entry.get("date", "")[:10],  # date portion only
            "change": entry.get("change"),
            "mrr_after": after.get("mrr"),
            "ndcg@5_after": after.get("ndcg@5"),
            "avg_judge_score": after.get("avg_judge_score"),
        }
    )

log_df = pd.DataFrame(log_rows)
log_df

There is one log entry so far: the 20-paper baseline grid. The `avg_judge_score` of 2.525/5 is lower than the perfect retrieval metrics suggest — that gap is expected when the LLM was answering questions about papers *outside* the 20-paper subset it was given context from. Answer quality will improve once the full 100-paper index is built.

> **Takeaway:** The iteration log is the experiment notebook — it connects config decisions to metric outcomes and makes every experiment reproducible. Without it, you quickly lose track of what you tried and why MRR changed between runs.

---
## 8. Visualizations

The pipeline generates six charts that summarise experiment results. They are stored in `visualizations/` and reproduced below with captions.

In [ ]:
viz_dir = PROJECT_ROOT / "visualizations"

charts = [
    {
        "filename": "metrics_heatmap.png",
        "caption": (
            "**Metrics heatmap.** MRR, NDCG@5, and Recall@5 across all 12 configurations. "
            "All values saturate at 1.0 on the 20-paper subset, confirming correct ranking "
            "but insufficient corpus size for discrimination."
        ),
    },
    {
        "filename": "dimension_impact.png",
        "caption": (
            "**Dimension impact.** Effect of each experimental axis (chunking, embedding, "
            "retrieval method) on retrieval latency. Latency is the only differentiating "
            "metric at this corpus size."
        ),
    },
    {
        "filename": "before_after.png",
        "caption": (
            "**Before / after.** Comparison of retrieval metrics and judge scores before "
            "and after the first pipeline iteration. Retrieval metrics were already at ceiling; "
            "judge scores reflect the corpus-mismatch constraint."
        ),
    },
    {
        "filename": "radar_generation.png",
        "caption": (
            "**Radar chart — generation quality.** The four judge dimensions (Relevance, "
            "Accuracy, Completeness, Citation Quality) plotted as a radar for the best "
            "configuration. Completeness is the weakest dimension."
        ),
    },
    {
        "filename": "latency_distribution.png",
        "caption": (
            "**Latency distribution.** Per-query retrieval time across all experiments. "
            "Hybrid retrieval adds ~30–50 ms over dense-only due to the BM25 scoring step, "
            "but stays well within interactive latency budgets."
        ),
    },
    {
        "filename": "fusion_sweep.png",
        "caption": (
            "**Fusion parameter sweep.** MRR as a function of the hybrid fusion weight α. "
            "α=0.6 (60% dense, 40% BM25) performs best on the 20-paper subset — "
            "BM25 provides meaningful signal on keyword-heavy ML abstracts."
        ),
    },
]

for chart in charts:
    path = viz_dir / chart["filename"]
    if not path.exists():
        print(f"[missing] {chart['filename']}")
        continue
    print(chart["caption"])
    display(Image(filename=str(path), width=750))
    print()

> **Takeaway:** Pre-generated charts make it fast to share results without re-running experiments. The key pattern across all charts is that retrieval quality is at ceiling on the 20-paper subset — latency and judge scores are the only currently meaningful signals.

---
## 9. Key Findings & Takeaways

### What we learned from the 20-paper baseline grid

**Sliding window chunking wins on recall.**  
Overlapping windows (size=10, step=5) prevent content from falling at a chunk boundary where neither adjacent chunk captures it fully. Fixed-size and recursive chunkers occasionally miss boundary-straddling answers; sliding windows do not.

**`all-mpnet-base-v2` outperforms `all-MiniLM-L6-v2` on technical text.**  
MiniLM-L6 is 5× faster to encode but trades off representation quality for speed. On dense, domain-specific ML abstracts, mpnet-base's larger capacity captures more semantic nuance. The latency difference (< 100 ms per query) is negligible at interactive scale.

**Hybrid retrieval (α=0.6) edges out dense-only.**  
BM25 scores anchor results to exact technical terms (model names, metric names, dataset names) that embedding similarity can blur. A 60/40 blend of dense and sparse scores captures both semantic and lexical relevance.

**Judge scores (avg 2.52/5) reflect corpus mismatch, not retrieval failure.**  
The 20-paper index was built on a subset; many evaluation queries referenced papers outside that subset. When the retrieved chunks don't contain the answer, the LLM produces a plausible but unsupported response — which the judge correctly penalises on completeness and accuracy. This is expected to improve substantially once the full 100-paper index is built.

### What's next

1. **Run the full 100-paper baseline grid** — 12 configs × 281 queries. MRR saturation will break and meaningful per-config differences will emerge.
2. **Add reranking experiments** — cross-encoder reranking (e.g. `ms-marco-MiniLM-L-6-v2`) on top of the best retrieval config to see if a second-stage model improves NDCG@5.
3. **Prompt-tune the generator** — target the completeness gap identified by the judge: instruct the model to explicitly address all aspects covered in the retrieved chunks.
4. **Log the 100-paper run as iteration 2** — record deltas in `iteration_log.jsonl` so the improvement is traceable.